# Create the L1 (tissue-level) label map

**Goal:** group every Allen structure id of the NextBrain ontology into one of 8 L1 tissue classes
and save it as `data/outputs/l1_names_mapping.yaml` (`class name -> list of allen ids`).

**Steps**
1. Load the ontology CSV.
2. Build the L1 map from the ontology column `L1_tissue`.
3. Save it.
4. Compare it with the older hand-made map `data/inputs/converted_16l.yaml`
   and list every structure on which the two maps disagree.

In [5]:
import pandas as pd
import yaml

## 0. Paths and L1 classes

`L1_CLASSES` maps **our class name** (used in the output yaml) to the **name used in the ontology column `L1_tissue`**.
Its order is also the order of the classes in the saved yaml.

In [6]:
ONTOLOGY_CSV       = "data/inputs/nextbrain_ontology_v1.csv"
REFERENCE_MAP_YAML = "data/inputs/converted_16l.yaml"      # old hand-made map, only used for comparison
OUTPUT_MAP_YAML    = "data/outputs/l1_names_mapping.yaml"

L1_CLASSES = {
    # our class name             : name in ontology "L1_tissue"
    "Background"                 : "Background",
    "White Matter"               : "Cerebral WM",
    "Gray Matter"                : "Cerebral GM",
    "Cerebellum White Matter"    : "Cerebellar WM",
    "Cerebellum Gray Matter"     : "Cerebellar GM",
    "Deep Gray Matter"           : "Deep GM",
    "Brainstem"                  : "Brainstem",
    "CSF"                        : "CSF / ventricles",
}

## 1. Load the ontology

In [7]:
ontology = pd.read_csv(ONTOLOGY_CSV)
ontology["allen_id"] = ontology["allen_id"].astype(int)

# allen id -> readable structure name, used later in the comparison table
id_to_name = dict(zip(ontology["allen_id"], ontology["leaf_name"]))

print("Structures in ontology:", ontology["allen_id"].nunique())
print("L1_tissue values      :", sorted(ontology["L1_tissue"].unique()))

Structures in ontology: 335
L1_tissue values      : ['Background', 'Brainstem', 'CSF / ventricles', 'Cerebellar GM', 'Cerebellar WM', 'Cerebral GM', 'Cerebral WM', 'Deep GM', 'Other']


## 2. Build the L1 map from the ontology

`ontology_map`: `class name -> list of allen ids`, taken directly from the `L1_tissue` column.

Then two manual fixes:
- **506** (`pineal_body`) has `L1_tissue = "Other"`, which is not one of our classes, so we put it in *Brainstem*.
- **3** is not a leaf of the ontology (it is the whole-cortex parent node), but we want it in *Gray Matter*.

In [8]:
ontology_map = {
    our_name: ontology.loc[ontology["L1_tissue"] == onto_name, "allen_id"].tolist()
    for our_name, onto_name in L1_CLASSES.items()
}

# ids whose L1_tissue is not one of our classes (expected: only 506, "Other")
assigned = {i for ids in ontology_map.values() for i in ids}
unassigned = ontology.loc[~ontology["allen_id"].isin(assigned), ["allen_id", "leaf_name", "L1_tissue"]]
print("Ontology structures without an L1 class (before the manual fixes):")
print(unassigned.to_string(index=False))

# manual fixes
ontology_map["Brainstem"].append(506)   # pineal_body, L1_tissue = "Other"
ontology_map["Gray Matter"].append(3)   # whole-cortex parent node, not in the ontology csv

Ontology structures without an L1 class (before the manual fixes):
 allen_id   leaf_name L1_tissue
      506 pineal_body     Other


**Sanity check:** every ontology id is assigned exactly once (+1 for the extra id 3).

In [9]:
all_ids = [i for ids in ontology_map.values() for i in ids]
assert len(all_ids) == len(set(all_ids)), "some id is in more than one class"
assert set(all_ids) == set(ontology["allen_id"]) | {3}, "ids missing or extra"

for name, ids in ontology_map.items():
    print(f"{name:<25} {len(ids):>4} ids")
print(f"{'TOTAL':<25} {len(all_ids):>4} ids  (= {ontology['allen_id'].nunique()} ontology ids + id 3)")

Background                   1 ids
White Matter                 8 ids
Gray Matter                106 ids
Cerebellum White Matter      2 ids
Cerebellum Gray Matter       6 ids
Deep Gray Matter           120 ids
Brainstem                   91 ids
CSF                          2 ids
TOTAL                      336 ids  (= 335 ontology ids + id 3)


## 3. Save the map

In [10]:
with open(OUTPUT_MAP_YAML, "w") as f:
    yaml.dump(ontology_map, f, sort_keys=False)
print("Saved", OUTPUT_MAP_YAML)

Saved data/outputs/l1_names_mapping.yaml


## 4. Compare with the old map `converted_16l.yaml`

The old yaml is stored as a *list* of one-key dicts under `Combined Structures`:
```yaml
Combined Structures:
- Background:
  - 0
- White Matter:
  - 7
```
so we merge that list into one dict `reference_map` (`class name -> list of allen ids`), with the same shape as `ontology_map`.

In [11]:
with open(REFERENCE_MAP_YAML) as f:
    raw = yaml.safe_load(f)

reference_map = {name: [int(i) for i in ids]
                 for item in raw["Combined Structures"]
                 for name, ids in item.items()}

assert set(reference_map) == set(ontology_map), "the two maps use different class names"
for name in L1_CLASSES:
    print(f"{name:<25} ontology: {len(ontology_map[name]):>4} ids   reference: {len(reference_map[name]):>4} ids")

Background                ontology:    1 ids   reference:    1 ids
White Matter              ontology:    8 ids   reference:    9 ids
Gray Matter               ontology:  106 ids   reference:  100 ids
Cerebellum White Matter   ontology:    2 ids   reference:    5 ids
Cerebellum Gray Matter    ontology:    6 ids   reference:    2 ids
Deep Gray Matter          ontology:  120 ids   reference:  120 ids
Brainstem                 ontology:   91 ids   reference:   28 ids
CSF                       ontology:    2 ids   reference:    1 ids


### Structures on which the two maps disagree

For each allen id we look up its class in both maps. One row per id where they differ:

| column | meaning |
|---|---|
| `class_in_ontology` | class in the new map (from the ontology) |
| `class_in_reference` | class in the old `converted_16l.yaml` |

`(not present)` means the id does not appear in that map at all.

In [12]:
NOT_PRESENT = "(not present)"

id_to_ontology_class  = {i: name for name, ids in ontology_map.items()  for i in ids}
id_to_reference_class = {i: name for name, ids in reference_map.items() for i in ids}

rows = []
for allen_id in sorted(set(id_to_ontology_class) | set(id_to_reference_class)):
    onto_cls = id_to_ontology_class.get(allen_id, NOT_PRESENT)
    ref_cls  = id_to_reference_class.get(allen_id, NOT_PRESENT)
    if onto_cls != ref_cls:
        rows.append({"allen_id": allen_id,
                     "leaf_name": id_to_name.get(allen_id, "?"),
                     "class_in_ontology": onto_cls,
                     "class_in_reference": ref_cls})

disagreements = pd.DataFrame(rows)
print(f"{len(disagreements)} of {len(set(id_to_ontology_class) | set(id_to_reference_class))} ids disagree")

100 of 336 ids disagree


**Summary:** how many ids move from each reference class to each ontology class.

In [13]:
summary = (disagreements
           .groupby(["class_in_reference", "class_in_ontology"])
           .size().rename("n_ids").reset_index()
           .sort_values("n_ids", ascending=False))
summary

,class_in_reference,class_in_ontology,n_ids
0,(not present),Brainstem,69
6,Deep Gray Matter,Gray Matter,11
8,Gray Matter,Deep Gray Matter,5
5,Cerebellum White Matter,Cerebellum Gray Matter,4
3,Brainstem,Deep Gray Matter,4
10,White Matter,Deep Gray Matter,2
1,(not present),Deep Gray Matter,1
2,Brainstem,Cerebellum White Matter,1
4,Brainstem,White Matter,1
7,Deep Gray Matter,White Matter,1


**Ids whose class changed** (they are in both maps, but in different classes).

In [14]:
changed = disagreements[(disagreements["class_in_ontology"]  != NOT_PRESENT) &
                        (disagreements["class_in_reference"] != NOT_PRESENT)]
pd.set_option("display.max_rows", None)
changed.sort_values(["class_in_reference", "class_in_ontology", "allen_id"]).reset_index(drop=True)

,allen_id,leaf_name,class_in_ontology,class_in_reference
0,611,white_matter_of_hindbrain,Cerebellum White Matter,Brainstem
1,315,medial_portion_of_sth,Deep Gray Matter,Brainstem
2,316,laterodorsal_portion_of_sth,Deep Gray Matter,Brainstem
3,321,lateroventral_portion_of_sth,Deep Gray Matter,Brainstem
4,843,infundibular_stalk,Deep Gray Matter,Brainstem
5,114,optic_nerve,White Matter,Brainstem
6,715,fastigial_(medial)_nucleus,Cerebellum Gray Matter,Cerebellum White Matter
7,721,dentate_(lateral)_nucleus,Cerebellum Gray Matter,Cerebellum White Matter
8,751,medial_interpositus_(globose)_nucleus,Cerebellum Gray Matter,Cerebellum White Matter
9,752,lateral_interpositus_(emboliform)_nucleus,Cerebellum Gray Matter,Cerebellum White Matter


**Ids that are in only one of the two maps.**

In [15]:
only_one = disagreements[(disagreements["class_in_ontology"]  == NOT_PRESENT) |
                         (disagreements["class_in_reference"] == NOT_PRESENT)]
only_one.sort_values(["class_in_reference", "class_in_ontology", "allen_id"]).reset_index(drop=True)

,allen_id,leaf_name,class_in_ontology,class_in_reference
0,386,root_of_oculomotor_nerve,Brainstem,(not present)
1,401,parabrachial_pigmented_nucleus,Brainstem,(not present)
2,403,cerebellothalamic_tract,Brainstem,(not present)
3,415,pontocerebellar_tract,Brainstem,(not present)
4,431,white_matter_of_midbrain,Brainstem,(not present)
5,436,interpeduncular_nucleus,Brainstem,(not present)
6,447,edinger-westphal_nucleus_(accessory_oculomotor...,Brainstem,(not present)
7,464,oculomotor_nucleus,Brainstem,(not present)
8,468,precommissural_nucleus,Brainstem,(not present)
9,469,interstitial_nucleus_of_posterior_commissure,Brainstem,(not present)


In [18]:
# load this json: /home/agustin/phd/synthseg_nextbrain/project/dev2/simplified_trees_and_labels/data/inptus/simplifiednxb/labels_to_remove.json
import json
json_path = "/home/agustin/phd/synthseg_nextbrain/project/dev2/simplified_trees_and_labels/data/inptus/simplifiednxb/labels_to_remove.json"
# load
with open(json_path) as f:
    labels_to_remove = json.load(f)
ids_removed = [int(i) for i in labels_to_remove]

ids_only_one = only_one["allen_id"].tolist()

# verify the ids in json that are not in ids_only_one
ids_not_in_only_one = [i for i in ids_removed if i not in ids_only_one]
print(f"{len(ids_not_in_only_one)} ids in json are not in only_one")
# print those ids 
if len(ids_not_in_only_one) > 0:
    print("ids in json but not in only_one:", ids_not_in_only_one)
# verify the ids in ids_only_one that are not in json
ids_not_in_json = [i for i in ids_only_one if i not in ids_removed]
print(f"{len(ids_not_in_json)} ids in only_one are not in json")
if len(ids_not_in_json) > 0:
    print("ids in only_one but not in json:", ids_not_in_json)


1 ids in json are not in only_one
ids in json but not in only_one: [50]
0 ids in only_one are not in json
